# Utilzie DMS

In [ ]:
###### Input variables ######
INPUT_META_DIR = "/home/psh/data/CAD6_GKR01I/meta"
XLSX_FILES = [
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_H1_Single.xlsx",
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_H2_Single.xlsx",
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_H3_Single.xlsx",
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_L1_Single.xlsx",
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_L2_Single.xlsx",
"/home/psh/BioBetter/GZR-78I_B7H3/dms/GZR-78I_L3_Single.xlsx",
]
REL_AFFINITY_THRESHOLD = 0.75

In [ ]:
######  Merge individual dms excel files to a single merged csv file  ######
import pandas as pd
import os
from pathlib import Path

dfs = []

for f in XLSX_FILES:
    df = pd.read_excel(f)

    # (선택) 어떤 파일에서 왔는지 표시하고 싶으면
    df["source_file"] = Path(f).stem

    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

out_path = os.path.join(os.path.dirname(XLSX_FILES[0]), "dms_merged.csv")
merged_df.to_csv(out_path, index=False)



In [ ]:
###### Filter dms data by affinity threshold ######
import pandas as pd
import io

# ---------------------------------------------------------
# 1. 설정 변수 (사용자가 지정)
# ---------------------------------------------------------
csv_file_path = out_path
heavy_chain_id = 'A'             # Heavy chain ID (예: 'H', 'A', 'B' 등)
light_chain_id = 'B'             # Light chain ID (예: 'L', 'B', 'C' 등)
# ---------------------------------------------------------

def parse_mutations(csv_input, threshold, h_id, l_id):
    # CSV 읽기 (파일 경로인 경우 pd.read_csv(csv_file_path) 사용)
    # 여기서는 테스트용 문자열을 읽습니다.
    if isinstance(csv_input, str) and "\n" in csv_input:
        df = pd.read_csv(io.StringIO(csv_input))
    else:
        df = pd.read_csv(csv_input)

    mutation_list = []

    for index, row in df.iterrows():
        # 1. Relative Affinity 값 읽기 및 필터링
        try:
            rel_affinity = float(row['Relative affinity'])
        except ValueError:
            continue # 숫자가 아니면 스킵

        # 조건: 음수 스킵, 정확히 1.0 스킵, threshold 미만 스킵
        if rel_affinity < 0:
            continue
        if rel_affinity == 1.0:
            continue
        if rel_affinity < threshold:
            continue

        # 2. CDR 열을 보고 Chain 결정
        cdr = str(row['CDR']).strip()
        mutated_chain = ""
        
        if cdr.startswith('H'):
            mutated_chain = h_id
        elif cdr.startswith('L'):
            mutated_chain = l_id
        else:
            continue # H나 L로 시작하지 않으면 스킵 (혹은 예외처리)

        # 3. Variant 파싱 ({original AA}{residue id}{mutated AA})
        # 예: G31A -> G, 31, A
        variant = str(row['Variant']).strip()
        
        if len(variant) < 3:
            continue # 형식이 맞지 않으면 스킵

        original_aa = variant[0]       # 첫 글자
        mutated_aa = variant[-1]       # 마지막 글자
        residue_id = variant[1:-1]     # 중간 나머지 (숫자)

        # 4. 문자열 조합 및 리스트 저장
        # 형식: {original AA}{mutated chain}{mutated residue id}{mutated AA}
        formatted_str = f"{original_aa}{mutated_chain}{residue_id}{mutated_aa}"
        mutation_list.append(formatted_str)

    return mutation_list

# ---------------------------------------------------------
# 실행 및 결과 출력
# ---------------------------------------------------------
results = parse_mutations(csv_file_path, REL_AFFINITY_THRESHOLD, heavy_chain_id, light_chain_id)


In [6]:
###### Incorporate selected mutations to meta data ######

from data import utils as du 
import copy 
import torch 
import re 
import data.residue_constants as rc
import pandas as pd 
import os 
import numpy as np
import glob 

pkl_file_path = pkl_path = glob.glob(os.path.join(INPUT_META_DIR, "**", "*.pkl"), recursive=True)[0]
meta_path = os.path.join(INPUT_META_DIR, "metadata.csv")
write_dir = os.path.dirname(meta_path)
new_meta_path = os.path.join(write_dir, "metadata_dms.csv")

cols = [
    "h1_start", "h1_end",
    "h2_start", "h2_end",
    "h3_start", "h3_end",
    "l1_start", "l1_end",
    "l2_start", "l2_end",
    "l3_start", "l3_end",
]

# 기존 CSV 로드
if os.path.exists(meta_path):
    df = pd.read_csv(meta_path)
    cdr_ranges = df.loc[0, cols].to_dict()
else:
    # 테스트용 더미 데이터 혹은 에러 처리
    raise FileNotFoundError(f"{meta_path} not found.")

def process_mutations(processed_file_path, mut, scaffold_idx):
    """
    Apply mutations (substitution, deletion, insertion) to protein features.
    Prioritizes substitutions first, then handles indels with metadata correction.
    """
    
    # 1. 데이터 로드
    processed_feats = du.read_pkl(processed_file_path)
    
    # 2. Tensor 변환 (CPU)
    for k, v in processed_feats.items():
        if not torch.is_tensor(v):
            processed_feats[k] = torch.tensor(v)
    
    updated_scaffold_idx = copy.deepcopy(scaffold_idx)

    # 3. Mutation 없음 처리
    if mut == 'No_Mutation':
        return _convert_to_standard_types(processed_feats, updated_scaffold_idx)

    # 4. Mutation 파싱
    raw_mutations = mut.split('_')
    parsed_mutations = []

    for m in raw_mutations:
        match = re.match(r"([a-zA-Z])([a-zA-Z0-9])(\d+)(.*)", m)
        if not match:
            print(f"Warning: Cannot parse mutation string {m}")
            continue
        
        aa_char, chain_char, res_id_str, type_str = match.groups()
        target_res_id = int(res_id_str)
        target_chain_idx = du.chain_str_to_int(chain_char)
        
        parsed_mutations.append({
            'aa_char': aa_char,
            'chain_idx': target_chain_idx,
            'res_id': target_res_id,
            'type': type_str,
            'orig_str': m
        })

    # 5. 정렬: Substitution 먼저, 그 다음 Indel (Indel 내부는 뒤에서부터 처리)
    # type이 'del'이나 'ins'가 아니면 Substitution으로 간주
    def sort_key(x):
        is_indel = (x['type'] in ['del', 'ins'])
        # (Indel여부 False먼저, Residue ID 내림차순)
        # False(=0) < True(=1) -> Subs 먼저 옴
        # -x['res_id'] -> 큰 숫자가 먼저 옴 (내림차순)
        return (is_indel, -x['res_id'])

    parsed_mutations.sort(key=sort_key)

    # 6. Mutation 적용
    for p_mut in parsed_mutations:
        target_res_id = p_mut['res_id']
        target_chain_idx = p_mut['chain_idx']
        type_str = p_mut['type']
        aa_char = p_mut['aa_char']

        # 현재 데이터에서 타겟 위치 찾기
        curr_chain = processed_feats['chain_index']
        curr_res = processed_feats['residue_index']
        
        mask_loc = (curr_chain == target_chain_idx) & (curr_res == target_res_id)
        idx_loc = torch.where(mask_loc)[0]
        
        if len(idx_loc) == 0:
            print(f"Warning: Target residue {p_mut['orig_str']} location not found.")
            continue
        
        idx = idx_loc[0].item()

        # ---------------------------------------------------------------------
        # Case A: Deletion
        # ---------------------------------------------------------------------
        if type_str == 'del':
            for key in processed_feats.keys():
                feat = processed_feats[key]
                processed_feats[key] = torch.cat([feat[:idx], feat[idx+1:]], dim=0)
            
            # Residue Index 당기기
            mask_update = (processed_feats['chain_index'] == target_chain_idx) & \
                          (processed_feats['residue_index'] > target_res_id)
            processed_feats['residue_index'][mask_update] -= 1

            # Scaffold Index 업데이트 (Deletion)
            for region_key in updated_scaffold_idx:
                val = updated_scaffold_idx[region_key]
                if val > idx:
                    updated_scaffold_idx[region_key] -= 1
                # 만약 경계값(val == idx)을 삭제했다면? 
                # (일반적으로 범위를 줄이는 방향으로 로직 추가 가능하나, 
                # 여기서는 단순 shift만 유지)

        # ---------------------------------------------------------------------
        # Case B: Insertion
        # ---------------------------------------------------------------------
        elif type_str == 'ins':
            inserted_aa_idx = rc.restype_order.get(aa_char, 20)
            insert_pos = idx + 1 # 삽입될 절대 인덱스 위치
            
            # 새 행 생성
            new_row = {}
            new_row['aatype'] = torch.tensor([inserted_aa_idx], dtype=processed_feats['aatype'].dtype)
            new_row['atom_mask'] = torch.tensor(rc.STANDARD_ATOM_MASK[inserted_aa_idx]).unsqueeze(0)
            
            # 메타데이터 생성 (임시 ID, 나중에 밀림)
            new_row['residue_index'] = torch.tensor([target_res_id + 1], dtype=processed_feats['residue_index'].dtype)
            new_row['chain_index'] = torch.tensor([target_chain_idx], dtype=processed_feats['chain_index'].dtype)
            new_row['bb_mask'] = torch.tensor([1], dtype=processed_feats['bb_mask'].dtype)
            
            # 좌표 0 초기화
            new_row['atom_positions'] = torch.zeros((1, 37, 3), dtype=processed_feats['atom_positions'].dtype)
            new_row['b_factors'] = torch.zeros((1, 37), dtype=processed_feats['b_factors'].dtype)
            new_row['bb_positions'] = torch.zeros((1, 3), dtype=processed_feats['bb_positions'].dtype)
            new_row['modeled_idx'] = torch.tensor([0], dtype=processed_feats['modeled_idx'].dtype)
            
            for k in processed_feats.keys():
                if k not in new_row:
                    shape = list(processed_feats[k].shape)
                    shape[0] = 1
                    new_row[k] = torch.zeros(shape, dtype=processed_feats[k].dtype)

            # 기존 Residue Index 밀기
            mask_update = (processed_feats['chain_index'] == target_chain_idx) & \
                          (processed_feats['residue_index'] > target_res_id)
            processed_feats['residue_index'][mask_update] += 1
            
            # 데이터 삽입
            for key in processed_feats.keys():
                curr_data = processed_feats[key]
                to_insert = new_row[key].to(curr_data.device)
                processed_feats[key] = torch.cat([
                    curr_data[:insert_pos],
                    to_insert,
                    curr_data[insert_pos:]
                ], dim=0)

            # --- Scaffold Index (CDR Range) 업데이트 로직 [수정됨] ---
            # CDR 확장을 위해 Boundary 조건을 체크합니다.
# -----------------------------------------------------------------
            # [수정된 로직] Scaffold Index (CDR Range) 업데이트
            # -----------------------------------------------------------------
            for region_key in updated_scaffold_idx:
                val = updated_scaffold_idx[region_key]
                
                # 1. Start Key에 대한 처리
                if region_key.endswith('_start'):
                    # (1) 삽입 위치보다 뒤에 있는 CDR들의 시작점은 뒤로 한 칸 밀림
                    if val > insert_pos:
                        updated_scaffold_idx[region_key] += 1
                    
                    # (2) val == insert_pos 인 경우 (CDR 바로 앞에서 삽입 or CDR 시작점 확장)
                    # CDR Start 위치에 삽입되면, 기존 Start 인덱스를 유지해야 
                    # 새로 들어온 잔기(insert_pos)가 CDR의 첫 번째 잔기로 포함됨.
                    # 따라서 값을 증가시키지 않음 (Pass).
                    elif val == insert_pos:
                        pass
                
                # 2. End Key에 대한 처리
                elif region_key.endswith('_end'):
                    # (1) 삽입 위치와 같거나 뒤에 있는 End 점들은 뒤로 한 칸 밀림
                    # (End 점은 해당 잔기를 포함하므로 insert_pos와 같아도 밀려야 함)
                    if val >= insert_pos:
                        updated_scaffold_idx[region_key] += 1
                    
                    # (2) C-term 확장 (CDR End 바로 뒤에 삽입된 경우)
                    # 예: CDR이 10~20(End)인데 21에 삽입됨.
                    # 기존 End(20) == insert_pos(21) - 1
                    # CDR을 확장하고 싶다면 End를 20 -> 21로 늘려야 함 (+1)
                    elif val == insert_pos - 1:
                        updated_scaffold_idx[region_key] += 1

        # ---------------------------------------------------------------------
        # Case C: Substitution
        # ---------------------------------------------------------------------
        else:
            mutated_aa_char = type_str
            mutated_aa_idx = rc.restype_order.get(mutated_aa_char, 20)
            
            processed_feats['aatype'][idx] = mutated_aa_idx
            processed_feats['atom_mask'][idx] = torch.tensor(rc.STANDARD_ATOM_MASK[mutated_aa_idx])

    return _convert_to_standard_types(processed_feats, updated_scaffold_idx)


def _convert_to_standard_types(feats, scaffold_idx):
    """Features 변환 (Tensor -> NumPy / Python int)"""
    final_feats = {}
    final_scaffold_idx = {}

    for k, v in feats.items():
        if torch.is_tensor(v):
            final_feats[k] = v.detach().cpu().numpy()
        else:
            final_feats[k] = np.array(v)

    for k, v in scaffold_idx.items():
        if torch.is_tensor(v):
            final_scaffold_idx[k] = int(v.item())
        elif isinstance(v, (np.integer, np.floating)):
            final_scaffold_idx[k] = int(v.item())
        else:
            final_scaffold_idx[k] = int(v)

    return final_feats, final_scaffold_idx

# -----------------------------------------------------------
# 실행부 예시 (results 리스트와 process_mutations 연동)
# -----------------------------------------------------------
# results 변수가 정의되어 있다고 가정 (예: metadata.csv의 insertion_codes를 파싱한 리스트 등)
# 아래는 예시 루프입니다.

new_rows_list = []

for mt in results:
    try:
        new_processed_feats, new_cdr_ranges = process_mutations(pkl_file_path, mt, cdr_ranges)

        save_filename = os.path.basename(pkl_file_path).replace(".pkl", f"_{mt}.pkl")
        save_path = os.path.join(write_dir, save_filename)
        du.write_pkl(save_path, new_processed_feats)

        row_data = df.iloc[0].to_dict()
        row_data.update(new_cdr_ranges)
        row_data['processed_path'] = save_path
        # row_data['mutation'] = mt # 필요 시 mutation 명시
        new_rows_list.append(row_data)
        
    except Exception as e:
        print(f"Error processing mutation {mt}: {e}")

if new_rows_list:
    new_df = pd.DataFrame(new_rows_list)
    final_df = pd.concat([df, new_df], ignore_index=True)
    final_df.to_csv(new_meta_path, index=False)
    print(f"All processing complete. Saved to {new_meta_path}")


All processing complete. Saved to /home/psh/data/CAD6_GKR01I/meta/metadata_dms.csv
